# PHTS + Discrete-Time Koopman + SPIL → TDS-Risk: Power-Grid Cascade Prediction
Full implementation of Section 3 (Eqs. 1–40): persistent-homology topological signature (**PHTS**), topology-conditioned **discrete-time** Koopman forecasting, forecast-conditioned spectral perturbation localisation (**SPIL**) and the integrated **TDS-Risk** estimator with alarm calibration, line ranking, ablations and runtime measurement.

**Data.** The modified IEEE-39 RMS files are not part of this notebook, so `C.synthetic=True` runs a built-in *stand-in* simulator (quasi-static DC cascade model, adaptive non-uniform time stamps, jumps at switching events, duplicated/non-finite records, event logs). It exercises every equation end to end but is **not** evidence for the paper's claims. Set `C.synthetic=False` and edit `parse_scenario` in the loader cell for the real files.

**Deliberate implementation choices (not in the text, flagged where used):** (i) the Koopman ridge penalises the deviation from persistence (`K = I + dK`) and the reconstruction carries the out-of-POD-subspace residual forward, because otherwise operating-point offsets of unseen scenarios are lost; (ii) POD modes are scaled to unit variance; (iii) `gamma` and class weights of the risk heads are chosen on validation loss; (iv) standardised evidence is clipped to ±5; (v) Betti-curve integrals use a fine grid (`K=300`), accurate to one grid step per breakpoint; (vi) the PMU feature-group ablation is not implemented.

## 0. Configuration

In [ ]:
import os, time, math, warnings
from types import SimpleNamespace
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ----------------------------------------------------------------------------------
# Hyper-parameters.  Values marked (synthetic value) in the manuscript keep that value
# unless the demo sampling rate forces a rescale (comments show the paper setting).
# ----------------------------------------------------------------------------------
C = SimpleNamespace(
    seed=0,
    synthetic=True,            # True: built-in IEEE-39-style simulator; False: load real files (Section 3)
    data_dir="./grid_data",    # used only when synthetic=False
    # --- sampling / timeline (Sec. 3.1-3.2)
    dt=0.05,                   # paper: 0.01 s (PMU rate). Demo uses 0.05 s so that the notebook runs in minutes.
    t_fault=1.0, t_clear_true=0.10, t_sim=15.0,
    n_ops=16,                  # operating points of the synthetic study (x 34 initiating lines)
    split=(0.70, 0.15, 0.15),
    n_hist=25, Th=2.0, Tmax=8.0,     # history samples, prediction horizon, max time after clearance (paper: 10 s)
    # --- resampling (Eq. 8)
    tail_q=0.01, kappa=5.0, eps_delta=1e-6,
    # --- stress / status (Eqs. 10-13)
    eps_s=1e-3, q_low=0.01, M_conf=3, eps_false=0.01,
    phi_grid=np.round(np.arange(0.05, 0.951, 0.05), 2),
    # --- affinity / settling (Eqs. 14, 16)
    beta_aff=2.0, eta_settle=0.08,
    # --- PHTS (Eqs. 18-24)
    smax=3.0, K=300, G=12, T_ref=0.10, d_settle=0.10, W_slope=8,
    # --- Koopman (Eqs. 27-32)
    r_cands=(4, 8, 12, 16), ridge_grid=(1e-3, 1e-2, 1e-1, 1.0, 10.0), energy=0.995, r_tol=0.05, n_min_pairs=400, Kh_sec=0.20,
    # --- SPIL (Eq. 33)
    eig_tol=1e-6,
    # --- TDS-Risk (Eqs. 6-7)
    gamma_grid=(1.0, 10.0, 100.0, 1000.0), alpha_f=0.05, alpha_p=0.10, delta_det=0.80,
    theta_grid=np.round(np.arange(0.05, 0.96, 0.01), 2),
    # --- demo cascade physics (synthetic simulator only)
    heat_k=0.8,
)
C.Kh = max(1, int(round(C.Kh_sec / C.dt)))
C.n_ref = int(math.ceil(C.T_ref / C.dt)) + 1
rng_main = np.random.default_rng(C.seed)
print("dt =", C.dt, "| Koopman horizon (samples) =", C.Kh, "| reference window (samples) =", C.n_ref)

## 1. Monitored subgraph and measurement layout (Eq. 1)

In [ ]:
# Monitored transmission-branch subgraph of the IEEE 39-bus network (34 lines, 28 buses; Sec. 3.2)
LINES = [(1,2),(1,39),(2,3),(2,25),(3,4),(3,18),(4,5),(4,14),(5,6),(5,8),(6,7),(6,11),(7,8),(8,9),
         (9,39),(10,11),(10,13),(13,14),(14,15),(15,16),(16,17),(16,19),(16,21),(16,24),(17,18),(17,27),
         (21,22),(22,23),(23,24),(25,26),(26,27),(26,28),(26,29),(28,29)]
NODES = sorted({b for e in LINES for b in e})
N, NL = len(NODES), len(LINES)
nid = {b: i for i, b in enumerate(NODES)}
EI = np.array([nid[a] for a, _ in LINES]); EJ = np.array([nid[b] for _, b in LINES])
EIl, EJl = EI.tolist(), EJ.tolist()
GEN_BUSES = [1, 3, 5, 7, 9, 10, 16, 22, 26, 28]           # buses with a monitored frequency field
GEN_IDX = np.array([nid[b] for b in GEN_BUSES])
# measurement vector layout, Eq. (1): [V | f | I | P | Q]
SL = dict(V=slice(0, N), F=slice(N, N + len(GEN_BUSES)), I=slice(38, 38 + NL),
          P=slice(38 + NL, 38 + 2 * NL), Q=slice(38 + 2 * NL, 38 + 3 * NL))
D = 38 + 3 * NL
GROUPS = ["V", "F", "I", "P", "Q"]

def comp_labels(status):
    parent = list(range(N))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for l in np.flatnonzero(status):
        a, b = find(EIl[l]), find(EJl[l])
        if a != b: parent[a] = b
    return np.array([find(i) for i in range(N)])

assert len(set(comp_labels(np.ones(NL)))) == 1, "subgraph must be connected"
assert NL - N + 1 == 7
print(f"{N} nodes, {NL} lines, cycle rank {NL-N+1}, measurement dimension D = {D}")

## 2. Scenario source
Synthetic simulator (stand-in) and the real-data adapter.

In [ ]:
# ------------------------------------------------------------------------------------
# Synthetic stand-in for the modified IEEE-39 RMS simulations (Sec. 3.1).  It is NOT the
# paper's data: it is a quasi-static DC-flow cascade model with thermal-style line trips,
# generator trips, measurement lag/oscillations, adaptive (non-uniform) time stamps,
# jumps at switching events and PMU-style noise.  Replace with real files in the next cell.
# ------------------------------------------------------------------------------------
def dc_flows(p, b, status):
    A = np.zeros((N, N)); w = b * status
    A[EI, EJ] = w; A[EJ, EI] = w
    Lm = np.diag(A.sum(1)) - A
    lab = comp_labels(status); theta = np.zeros(N)
    for c in np.unique(lab):
        ix = np.flatnonzero(lab == c)
        if len(ix) == 1: continue
        pc = p[ix] - p[ix].mean()
        theta[ix] = np.linalg.solve(Lm[np.ix_(ix, ix)] + np.ones((len(ix), len(ix))) / len(ix), pc)
    return status * b * (theta[EI] - theta[EJ])

def make_op(o, seed):
    r = np.random.default_rng(seed * 1000 + o)
    p = r.normal(0, 1, N); p -= p.mean(); p *= r.uniform(0.9, 1.6)
    b = 1.0 / r.uniform(0.04, 0.25, NL)
    f0 = dc_flows(p, b, np.ones(NL))
    return dict(p=p, b=b, cap=np.abs(f0) * r.uniform(2.0, 3.4, NL) + r.uniform(0.25, 0.7, NL),
                qc=r.uniform(0.08, 0.20, NL), V0=r.uniform(0.98, 1.04, N),
                ramp=0.0 if r.random() < 0.35 else r.uniform(0.005, 0.035),
                gen_lim=r.uniform(1.5, 3.0), p_gen_after=r.uniform(0.0, 0.6))

def simulate_cascade(op, init, rng):
    h, t_f = 0.01, C.t_fault; t_c = t_f + C.t_clear_true
    status = np.ones(NL); heat = np.zeros(NL); p = op["p"].copy()
    events, segs = [], [(0.0, status.copy(), p.copy())]
    f_topo = dc_flows(p, op["b"], status)
    cleared = gen_done = False; t_first = gen_sched = None; n_after = 0; t_end = C.t_sim; t = 0.0
    if rng.random() < 0.02: events.append(("G9", t_f + 0.05))      # rare entry before clearance -> Omega_x
    while t < t_end:
        t += h; changed = False
        if (not cleared) and t >= t_c:
            cleared = True; status[init] = 0; events.append((f"L{init}", t_c)); changed = True
        if cleared:
            g = 1 + op["ramp"] * (t - t_c)
            ratio = np.abs(f_topo) * g / op["cap"] * status
            heat = np.where(ratio > 1, heat + h * C.heat_k * np.maximum(ratio - 1, 0), heat * 0.97)
            trip = (status > 0) & ((heat >= 1.0) | (ratio > 2.2))
            if trip.any():
                for l in np.flatnonzero(trip):
                    status[l] = 0; heat[l] = 0; events.append((f"L{l}", t)); n_after += 1
                changed = True
                if t_first is None:
                    t_first = t
                    if rng.random() < op["p_gen_after"]: gen_sched = t + rng.uniform(0.3, 2.0)
            if (not gen_done) and (g > op["gen_lim"] or (gen_sched is not None and t >= gen_sched)):
                gen_done = True; k = int(np.argmax(p)); p[k] *= 0.4
                events.append((f"G{k}", t)); n_after += 1; changed = True
                if t_first is None: t_first = t
        if changed:
            f_topo = dc_flows(p, op["b"], status)
            segs.append((t, status.copy(), p.copy()))
            if n_after >= 5 and t_end == C.t_sim: t_end = t + 1.0
    return dict(events=sorted(events, key=lambda e: e[1]), segs=segs, t_end=min(t_end, C.t_sim))

INC = np.zeros((N, NL)); INC[EI, np.arange(NL)] = 1; INC[EJ, np.arange(NL)] = 1

def synth_record(op, init, sim, rng):
    segs = sim["segs"]; seg_t = np.array([s[0] for s in segs])
    t_f, t_c = C.t_fault, C.t_fault + C.t_clear_true
    ev_t = sorted(set([t_f, t_c] + [s[0] for s in segs[1:]] + [e[1] for e in sim["events"]]))
    base = C.dt * 0.4; t = [0.0]                                    # adaptive, non-uniform native steps
    while t[-1] < sim["t_end"]:
        near = any(0 <= t[-1] - e < 0.1 for e in ev_t)
        t.append(t[-1] + base * (0.1 if near else rng.uniform(0.6, 1.4)))
    t = np.array(t); t = t[t <= sim["t_end"]]; T = len(t)
    idx = np.searchsorted(seg_t, t, side="right") - 1
    Fseg = np.array([dc_flows(s[2], op["b"], s[1]) for s in segs]); Sseg = np.array([s[1] for s in segs])
    g = np.where(t >= t_c, 1 + op["ramp"] * (t - t_c), 1.0)
    target = Fseg[idx] * g[:, None]
    Flag = np.zeros((T, NL)); Flag[0] = target[0]
    for k in range(1, T):
        a = np.exp(-(t[k] - t[k - 1]) / 0.25)
        Flag[k] = target[k] + (Flag[k - 1] - target[k]) * a
    osc = np.zeros((T, NL)); oscV = np.zeros((T, N)); oscF = np.zeros((T, len(GEN_IDX)))
    for k in range(1, len(segs)):
        te = seg_t[k]; m = (t >= te); dtau = np.maximum(t - te, 0)
        env = np.exp(-dtau / 0.6) * np.sin(2 * np.pi * 1.3 * dtau) * m
        osc += env[:, None] * ((Fseg[k] - Fseg[k - 1]) * 0.5)[None, :]
        oscV += env[:, None] * rng.normal(0, 0.015, N)[None, :]
        oscF += (np.exp(-dtau / 1.0) * np.sin(2 * np.pi * 0.8 * dtau) * m)[:, None] * rng.normal(0, 0.0015, len(GEN_IDX))[None, :]
    u = Sseg[idx]; F = u * (Flag + osc); ratio = np.abs(F) / op["cap"]
    V = op["V0"][None, :] - 0.012 * (ratio ** 2) @ INC.T + oscV
    fm = np.flatnonzero((t >= t_f) & (t < t_c))
    V[np.ix_(fm, [EI[init], EJ[init]])] -= 0.4
    V = np.clip(V, 0.3, 1.2)
    P = F.copy(); Q = u * (op["qc"][None, :] + 0.25 * F ** 2 / op["cap"][None, :])
    P[fm, init] *= 0.2
    I = np.hypot(P, Q) / V[:, EI]; I[fm, init] += 3.0; I *= u; I[fm, init] = (np.hypot(P, Q) / V[:, EI] + 3.0)[fm, init]
    gstep = np.zeros(T)
    for c, te in sim["events"]:
        if c.startswith("G"): gstep += -0.01 * np.exp(-np.maximum(t - te, 0) / 3.0) * (t >= te)
    Fr = 1.0 - 0.003 * (g - 1)[:, None] + oscF + gstep[:, None] * rng.uniform(0.6, 1.0, len(GEN_IDX))[None, :]
    V += rng.normal(0, 1e-4, V.shape); Fr += rng.normal(0, 3e-6, Fr.shape)
    I += rng.normal(0, 3e-4, I.shape); P += rng.normal(0, 3e-4, P.shape); Q += rng.normal(0, 3e-4, Q.shape)
    X = np.hstack([V, Fr, I, P, Q])
    if rng.random() < 0.30:                                         # duplicated time stamp (collapsed in Sec. 3.2)
        k = int(rng.integers(5, T - 5)); t = np.insert(t, k, t[k]); X = np.insert(X, k, X[k] + 1e-3, axis=0)
    if rng.random() < 0.30:                                         # terminated run: non-finite tail records
        t = np.r_[t, t[-1] + 0.01 * np.arange(1, 4)]; X = np.vstack([X, np.full((3, D), np.nan)])
    return t, X.astype(np.float64)

def build_synthetic():
    ops = [make_op(o, C.seed) for o in range(C.n_ops)]; raw = []
    for o in range(C.n_ops):
        for l in range(NL):
            rng = np.random.default_rng(C.seed * 100003 + o * 101 + l)
            sim = simulate_cascade(ops[o], l, rng); t, X = synth_record(ops[o], l, sim, rng)
            raw.append(dict(sid=f"op{o:02d}_L{l:02d}", op=o, t=t, X=X, events=sim["events"]))
    return raw

In [ ]:
# ------------------------------------------------------------------------------------
# Real-data adapter.  Every scenario must become a dict:
#   sid    : "op<k>_L<line>"  (operating-point index and initiating line; the line index is
#            read from the identifier, Sec. 3.1)
#   op     : operating-point index (used for the 70/15/15 split)
#   t, X   : native time stamps (T,) and RMS records (T, D) in the Eq. (1) order [V|f|I|P|Q]
#   events : list of (component, time), e.g. ("L7", 1.10), ("G3", 2.40) ; line outages are "L<index 0..33>"
# Edit `parse_scenario` to the column names / file layout of the real simulation files.
# ------------------------------------------------------------------------------------
def parse_scenario(csv_path, event_path, sid):
    df = pd.read_csv(csv_path)
    t = df["time"].to_numpy(float)
    # TODO map real column names to the PMU-compatible subset (59 V, 17 f, 34x I/P/Q in the paper);
    # the demo layout uses D = 38 + 3*34 fields, so adapt SL/D in the topology cell for the real layout.
    X = df.drop(columns=["time"]).to_numpy(float)
    ev = pd.read_csv(event_path)                         # columns: component, time
    events = [(str(c), float(tt)) for c, tt in zip(ev["component"], ev["time"])]
    op = int(sid.split("_L")[0].replace("op", ""))
    return dict(sid=sid, op=op, t=t, X=X, events=events)

def load_real(root):
    raw = []
    for f in sorted(os.listdir(os.path.join(root, "records"))):
        sid = f.replace(".csv", "")
        raw.append(parse_scenario(os.path.join(root, "records", f), os.path.join(root, "events", f), sid))
    return raw

t0 = time.time()
raw = build_synthetic() if C.synthetic else load_real(C.data_dir)
print(f"{len(raw)} scenarios loaded in {time.time()-t0:.1f}s")

## 3. Cleaning, resampling (Eq. 8), normalisation (Eq. 9) and operating-point split (Sec. 3.1–3.2)

In [ ]:
# ===== Sec. 3.2: cleaning, uniform resampling (Eq. 8), normalisation (Eq. 9) =====
def clean_native(t, X):
    ok = np.isfinite(X).all(1) & np.isfinite(t)
    if not ok.any(): return None
    last = np.flatnonzero(ok)[-1]                          # truncate a terminated run at its last finite record
    t, X, ok = t[:last + 1], X[:last + 1], ok[:last + 1]
    t, X = t[ok], X[ok]
    o = np.argsort(t, kind="stable"); t, X = t[o], X[o]
    keep = np.r_[t[1:] != t[:-1], True]                    # duplicated time stamps -> last record
    return t[keep], X[keep]

def resample(t, X):
    d = np.abs(np.diff(X, axis=0))
    delta = np.quantile(d, 1 - C.tail_q, axis=0) + C.eps_delta          # scale of a normal increment
    cont = (d <= C.kappa * delta).all(1)                                # continuous native intervals
    grid = np.arange(t[0], t[-1] + 1e-12, C.dt)
    k = np.clip(np.searchsorted(t, grid, side="right") - 1, 0, len(t) - 2)
    w = np.clip((grid - t[k]) / (t[k + 1] - t[k]), 0, 1)
    lin = X[k] + w[:, None] * (X[k + 1] - X[k])
    Xg = np.where(cont[k][:, None], lin, X[k])                          # hold across jumps (switching events)
    return grid, Xg, float(np.mean(t[k + 1] - grid))                    # emission delay <= one native step

def line_signals(Xp):
    V = Xp[:, SL["V"]]; I = Xp[:, SL["I"]]
    return V, I, np.hypot(Xp[:, SL["P"]], Xp[:, SL["Q"]])

def init_scenarios(raw):
    S = []
    for r in raw:
        cl = clean_native(np.asarray(r["t"], float), np.asarray(r["X"], float))
        if cl is None or len(cl[0]) < 10: continue
        tg, Xg, dl = resample(*cl)
        S.append(SimpleNamespace(sid=r["sid"], op=r["op"], init_id=int(r["sid"].split("_L")[1]), t=tg, Xp=Xg,
                                 events=r["events"], emit_delay=dl, n_native=len(cl[0])))
    return S

scen = init_scenarios(raw)
ops = np.array(sorted({s.op for s in scen})); perm = np.random.default_rng(C.seed).permutation(ops)
n_tr = int(round(C.split[0] * len(ops))); n_va = int(round(C.split[1] * len(ops)))
split_of_op = {}
for i, o in enumerate(perm): split_of_op[o] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")
for s in scen: s.split = split_of_op[s.op]          # all scenarios sharing an operating point share a subset
train = [s for s in scen if s.split == "train"]; val = [s for s in scen if s.split == "val"]; test = [s for s in scen if s.split == "test"]
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()},
      "| mean emission delay (s):", np.mean([s.emit_delay for s in scen]).round(4))

# global statistics from TRAINING data only
Xtr = np.vstack([s.Xp for s in train])
G = SimpleNamespace(mu=Xtr.mean(0), sd=Xtr.std(0))
G.inv = np.where(G.sd > 1e-9, 1.0 / np.maximum(G.sd, 1e-12), 0.0)       # constant fields map to zero (Eq. 9)
Vpre = np.vstack([line_signals(s.Xp[s.t < C.t_fault - 0.1])[0] for s in train])
Ipre = np.vstack([line_signals(s.Xp[s.t < C.t_fault - 0.1])[1] for s in train])
Spre = np.vstack([line_signals(s.Xp[s.t < C.t_fault - 0.1])[2] for s in train])
G.V_ref = Vpre.mean(0)                                  # per-bus pre-fault voltage reference
G.S_ref = Spre.mean(0)                                  # per-line pre-fault apparent-power reference (Eq. 10)
G.I_low = np.quantile(Ipre, C.q_low, axis=0)            # low in-service levels (Eq. 11)
G.S_low = np.quantile(Spre, C.q_low, axis=0)
for s in scen: s.Xn = (s.Xp - G.mu) * G.inv             # Eq. (9)
print("normalised state ready; S_ref range", G.S_ref.min().round(3), G.S_ref.max().round(3))

## 4. Stress (Eq. 10), line status (Eqs. 11–13), clearance (Eq. 17), switching/settling indicators (Eq. 16), reference window (Eq. 20), event-log labels (Eq. 2)

In [ ]:
# ===== Sec. 3.2: stress (Eq. 10), line status (Eqs. 11-13), clearance (Eq. 17), indicators (Eq. 16) =====
def confirm_status(raw_in, M):
    """Eq. (12): switch to 0 / 1 only after M consecutive raw samples agree, otherwise hold (causal)."""
    T, L = raw_in.shape
    cs = np.vstack([np.zeros((1, L)), np.cumsum(raw_in, 0)])
    win = cs[M:] - cs[:-M]                                # sum of raw over samples n-M+1..n  (n >= M-1)
    ev = -np.ones((T, L), int)
    sub = ev[M - 1:]; sub[win == 0] = 0; sub[win == M] = 1
    idx = np.where(ev >= 0, np.arange(T)[:, None], -1)
    idx = np.maximum.accumulate(idx, axis=0)
    return np.where(idx >= 0, np.take_along_axis(ev, np.maximum(idx, 0), axis=0), 1).astype(np.int8)

def raw_status(I, S, phi):
    return (~((I < phi * G.I_low) & (S < phi * G.S_low))).astype(np.int8)       # Eq. (11)

def logged_outage_times(sc):
    tout = np.full(NL, np.inf)
    for c, te in sc.events:
        if c.startswith("L"): tout[int(c[1:])] = min(tout[int(c[1:])], te)
    return tout

# --- Eq. (13): select phi on training scenarios against the event log
cache = []
for sc in train:
    V, I, S = line_signals(sc.Xp); cache.append((I, S, logged_outage_times(sc), sc.t))
rows = []
for phi in C.phi_grid:
    fa = nfa = ms = nms = 0
    for I, S, tout, t in cache:
        st = confirm_status(raw_status(I, S, phi), C.M_conf)
        inserv = t[:, None] < tout[None, :]; chk = t[:, None] >= tout[None, :] + C.M_conf * C.dt
        fa += ((st == 0) & inserv).sum(); nfa += inserv.sum(); ms += ((st == 1) & chk).sum(); nms += chk.sum()
    rows.append((phi, fa / nfa, ms / max(nms, 1)))
calib = pd.DataFrame(rows, columns=["phi", "false_disc_rate", "missed_rate"])
feas = calib[calib.false_disc_rate <= C.eps_false]
phi_star = float((feas if len(feas) else calib).sort_values(["missed_rate", "false_disc_rate"]).iloc[0].phi)
G.phi = phi_star
delays = []
for I, S, tout, t in cache:
    st = confirm_status(raw_status(I, S, phi_star), C.M_conf)
    for l in np.flatnonzero(np.isfinite(tout) & (tout < t[-1])):
        h = np.flatnonzero((t >= tout[l]) & (st[:, l] == 0))
        if len(h): delays.append(t[h[0]] - tout[l])
print(f"phi* = {phi_star} | false-disc. = {calib.set_index('phi').loc[phi_star,'false_disc_rate']:.4f} | "
      f"missed = {calib.set_index('phi').loc[phi_star,'missed_rate']:.4f} | median detection delay = {np.median(delays):.3f}s "
      f"(p90 {np.quantile(delays,0.9):.3f}s)")

# --- per-scenario signals, labels, clearance and indicators
def label_info(sc):
    ini = f"L{sc.init_id}"
    t_init = min([t for c, t in sc.events if c == ini], default=np.nan)
    others = [(c, t) for c, t in sc.events if c != ini]
    sc.t_clear_log = t_init
    sc.omega_x = bool(np.isnan(t_init) or any(t < t_init for c, t in others))            # excluded set
    sc.tau = min([t for c, t in others], default=np.inf)                                   # Eq. (2)
    sc.line_events = [(int(c[1:]), t) for c, t in others if c.startswith("L")]

def prep(sc):
    label_info(sc)
    V, I, S = line_signals(sc.Xp)
    sc.V = V; sc.rho = S / (G.S_ref[None, :] + C.eps_s)                                    # Eq. (10)
    sc.status = confirm_status(raw_status(I, S, G.phi), C.M_conf)                          # Eqs. (11)-(12)
    T = len(sc.t); st = sc.status
    sc.s_ind = np.zeros(T, np.int8); sc.s_ind[1:] = (st[1:] != st[:-1]).any(1)             # Eq. (16) switching
    sc.sigma = np.zeros(T, np.int8); sc.sigma[1:] = (np.abs(np.diff(sc.Xn, axis=0)).max(1) <= C.eta_settle)  # settling
    # Eq. (17): first confirmed opening after the fault
    nf = int(np.searchsorted(sc.t, C.t_fault)); drop = (st[:-1] == 1) & (st[1:] == 0); drop[:max(nf - 1, 0)] = False
    rws = np.flatnonzero(drop.any(1)); sc.n_c = sc.l_hat = sc.t_c_hat = None
    if len(rws):
        n = rws[0] + 1; ls = np.flatnonzero(drop[rws[0]])
        sc.n_c = int(n); sc.l_hat = int(ls[np.argmax(G.S_ref[ls])]); sc.t_c_hat = float(sc.t[n])
        sc.n_end = int(min(T - 1, np.searchsorted(sc.t, sc.t_c_hat + C.Tmax, side="right") - 1))
    # Eq. (20): reference window = first run of n_ref switching-free, settled samples after clearance + settling delay
    sc.ref = None
    if sc.n_c is not None:
        n0 = int(np.searchsorted(sc.t, sc.t_c_hat + C.d_settle)); ok = ((sc.s_ind == 0) & (sc.sigma == 1)); ok[:n0] = False
        cs = np.convolve(ok.astype(int), np.ones(C.n_ref, int), "valid"); hit = np.flatnonzero(cs == C.n_ref)
        if len(hit): sc.ref = (int(hit[0]), int(hit[0] + C.n_ref - 1))
for sc in scen: prep(sc)

audit = pd.DataFrame([dict(split=sp, scenarios=len(ss), omega_x=sum(s.omega_x for s in ss),
                           no_clearance=sum((s.n_c is None) for s in ss if not s.omega_x),
                           cascading=sum(np.isfinite(s.tau) for s in ss if not s.omega_x),
                           stable=sum(np.isinf(s.tau) for s in ss if not s.omega_x),
                           ref_coverage=np.mean([s.ref is not None for s in ss if not s.omega_x and s.n_c is not None]),
                           init_line_agree=np.mean([s.l_hat == s.init_id for s in ss if not s.omega_x and s.n_c is not None]))
                      for sp, ss in [("train", train), ("val", val), ("test", test)]]).set_index("split")
print(audit.round(3))
# usable scenarios: not in Omega_x and with a confirmed clearance
for ss in (train, val, test): pass
train_u = [s for s in train if not s.omega_x and s.n_c is not None]
val_u = [s for s in val if not s.omega_x and s.n_c is not None]
test_u = [s for s in test if not s.omega_x and s.n_c is not None]

## 5. PHTS — clearance-conditioned persistent-homology signature (Eqs. 18–24, Props. 1–2)

In [ ]:
# ===== Sec. 3.3  PHTS: clearance-conditioned persistent-homology topological signature =====
KG = np.arange(C.K) * (C.smax / C.K); DS = C.smax / C.K                 # fine filtration grid on [0, smax)
GIDX = ((np.arange(C.G) + 0.5) * C.K / C.G).astype(int)                 # G stress levels for Eq. (24)

def kruskal(w, act):
    """Minimum spanning forest of the in-service graph (ties by line index). Returns tree/non-tree line ids."""
    parent = list(range(N)); tree, non = [], []
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for l in np.argsort(w, kind="stable"):
        if not act[l]: continue
        a, b = find(EIl[l]), find(EJl[l])
        if a != b: parent[a] = b; tree.append(int(l))
        else: non.append(int(l))
    return tree, non

def assign_random(w, act, kappa):
    """Edge-permutation control: same multiset of stresses, randomly re-assigned to the in-service lines."""
    ins = np.flatnonzero(act); vals = np.sort(w[ins]); out = np.zeros_like(w)
    out[ins[np.argsort(kappa[ins], kind="stable")]] = vals
    return out

def betti_curves(w, act, kappa=None):
    """Eq. (18)-(19): exact step functions beta0(s), beta1(s) on the fine grid (right-continuous)."""
    w = np.minimum(w, C.smax)
    if kappa is not None: w = assign_random(w, act, kappa)
    tree, non = kruskal(w, act)
    b0 = N - np.searchsorted(np.sort(w[tree]), KG, side="right")          # component deaths = MSF stresses
    b1 = np.searchsorted(np.sort(w[non]), KG, side="right")               # cycle births = remaining stresses
    return b0.astype(float), b1.astype(float)

def connectivity_impacts(w, act):
    """Eq. (23): c_l = r_l - w_l for lines of the minimum spanning forest, 0 otherwise."""
    w = np.minimum(w, C.smax); tree, non = kruskal(w, act); c = np.zeros(NL)
    adj = {i: [] for i in range(N)}
    for l in tree: adj[EIl[l]].append((EJl[l], l)); adj[EJl[l]].append((EIl[l], l))
    for e in tree:
        side = {EIl[e]}; stack = [EIl[e]]
        while stack:
            u = stack.pop()
            for v, l in adj[u]:
                if l != e and v not in side: side.add(v); stack.append(v)
        r = min([w[l] for l in non if (EIl[l] in side) != (EJl[l] in side)], default=C.smax)
        c[e] = r - w[e]
    return c

def phts_signature(sc, kappa=None):
    """Eqs. (20)-(24). Returns z (T, 4+2G): NaN before the reference is ready."""
    T = len(sc.t); z = np.full((T, 4 + 2 * C.G), np.nan)
    if sc.ref is None: return z
    i0, i1 = sc.ref
    cur = [betti_curves(sc.rho[n], sc.status[n], kappa) for n in range(i0, i1 + 1)]
    B0r = np.mean([c[0] for c in cur], 0); B1r = np.mean([c[1] for c in cur], 0)      # Eq. (20)
    h0, h1 = [], []; x = np.arange(C.W_slope) * C.dt; xc = x - x.mean()
    for n in range(i1, sc.n_end + 1):
        b0, b1 = betti_curves(sc.rho[n], sc.status[n], kappa)
        dv0, dv1 = b0 - B0r, B1r - b1
        d0 = np.maximum(dv0, 0).sum() * DS; d1 = np.maximum(dv1, 0).sum() * DS          # Eq. (21)
        h0.append(d0); h1.append(d1)
        if len(h0) >= C.W_slope:                                                        # Eq. (22): causal LS slope
            s0 = (xc * (np.array(h0[-C.W_slope:]) - np.mean(h0[-C.W_slope:]))).sum() / (xc ** 2).sum()
            s1 = (xc * (np.array(h1[-C.W_slope:]) - np.mean(h1[-C.W_slope:]))).sum() / (xc ** 2).sum()
        else: s0 = s1 = 0.0
        z[n] = np.r_[d0, d1, max(s0, 0), max(s1, 0), dv0[GIDX], dv1[GIDX]]               # Eq. (24)
    return z

# ---- numerical verification of Eq. (23) and of the identity behind Proposition 2
rr = np.random.default_rng(1); err23 = err26 = 0
for _ in range(200):
    w = rr.uniform(0, C.smax, NL); act = (rr.random(NL) > 0.1).astype(int); c = connectivity_impacts(w, act)
    b0, b1 = betti_curves(w, act)
    for l in np.flatnonzero(c > 0)[:3]:                                  # removal of a forest line: area(beta0) grows by c_l
        a2 = act.copy(); a2[l] = 0; b0x, _ = betti_curves(w, a2)
        err23 = max(err23, abs((b0x - b0).sum() * DS - c[l]))
    # Prop. 2 identity: area(beta0) - area(beta1) = N*smax - sum_l (smax - w_l) over in-service lines,
    # i.e. it depends on the stresses only through their empirical distribution (not on which line carries them)
    lhs = (b0 - b1).sum() * DS; rhs = N * C.smax - (C.smax - np.minimum(w, C.smax))[act > 0].sum()
    err26 = max(err26, abs(lhs - rhs))
print(f"Eq. (23) check: max |area change - c_l| = {err23:.4f} | Eq. (26) identity gap = {err26:.4f} (grid step {DS:.3f} per breakpoint)")

t0 = time.time()
for sc in scen: sc.z_phts = None
for sc in train_u + val_u + test_u: sc.z_phts = phts_signature(sc)
print(f"PHTS computed for {len(train_u)+len(val_u)+len(test_u)} scenarios in {time.time()-t0:.1f}s")

## 6. Topology-conditioned discrete-time Koopman forecasting (Eqs. 27–32)

In [ ]:
# ===== Sec. 3.4  Topology-conditioned discrete-time Koopman forecasting (EDMD with POD + polynomial dictionary) =====
def lift(xi):                                              # Eq. (28): [1, xi, vech(xi xi^T)]
    il, jl = np.tril_indices(xi.shape[-1])
    return np.concatenate([np.ones(xi.shape[:-1] + (1,)), xi, xi[..., il] * xi[..., jl]], axis=-1)

def collect_pairs(scs):
    by = {}
    for sc in scs:
        idx = np.arange(sc.n_c, sc.n_end)
        idx = idx[(sc.s_ind[idx] == 0) & (sc.s_ind[idx + 1] == 0)]       # no switching inside a training pair
        if len(idx): by.setdefault(sc.l_hat, []).append((sc.Xn[idx], sc.Xn[idx + 1]))
    return {l: (np.vstack([a for a, _ in v]), np.vstack([b for _, b in v])) for l, v in by.items()}

nc_train = [s for s in train_u if np.isinf(s.tau)]; nc_val = [s for s in val_u if np.isinf(s.tau)]
pairs = collect_pairs(nc_train)
Zall = np.vstack([np.vstack(v) for v in pairs.values()])
pod_mu = Zall.mean(0)
_, sv, Vt = np.linalg.svd(Zall - pod_mu, full_matrices=False)
pod_U = Vt.T; pod_sd = sv / np.sqrt(len(Zall)); energy = np.cumsum(sv ** 2) / np.sum(sv ** 2)
r_energy = int(np.searchsorted(energy, C.energy) + 1)
print(f"POD: {len(Zall)} snapshots from {len(nc_train)} non-cascading training scenarios, {len(pairs)} initiating-line classes; "
      f"{C.energy:.1%} energy needs r = {r_energy}")

def build_bank(r, ridge):
    U, sd = pod_U[:, :r], pod_sd[:r]; m = 1 + r + r * (r + 1) // 2
    xi = lambda X: ((X - pod_mu) @ U) / sd                          # Eq. (27) (modes scaled to unit variance for conditioning)
    sums = {}
    for l, (Xa, Xb) in pairs.items():
        pa, pb = lift(xi(Xa)), lift(xi(Xb)); sums[l] = (pa.T @ pa, pb.T @ pa, len(Xa))
    Gp = sum(v[0] for v in sums.values()); Ap = sum(v[1] for v in sums.values())
    # Eq. (29), ridge shrinking towards persistence (K = I + dK, only dK is penalised): lambda -> inf gives the persistence forecast
    solve = lambda Gm, Am: np.eye(m) + np.linalg.solve(Gm + ridge * np.trace(Gm) / m * np.eye(m), (Am - Gm).T).T
    Kp = solve(Gp, Ap)
    Ks = {l: (solve(Gm, Am) if n >= C.n_min_pairs else Kp) for l, (Gm, Am, n) in sums.items()}
    return dict(r=r, ridge=ridge, U=U, sd=sd, Kp=Kp, Ks=Ks, n_class_models=sum(n >= C.n_min_pairs for _, _, n in sums.values()))

def forecast(bank, X, l, steps):
    """Eq. (30): multi-step forecast in normalised coordinates, re-lifting the predicted reduced state each step."""
    K = bank["Ks"].get(l, bank["Kp"]); r = bank["r"]
    xi0 = ((X - pod_mu) @ bank["U"]) / bank["sd"]; xi = xi0
    for _ in range(steps):
        xi = np.clip((lift(xi) @ K.T)[:, 1:1 + r], -10, 10)           # numerical safeguard against transient blow-up
    # reconstruction: the POD-subspace part follows the Koopman forecast, the out-of-subspace residual of the current
    # sample is carried forward unchanged (otherwise operating-point specific offsets of unseen scenarios are lost)
    return X + ((xi - xi0) * bank["sd"]) @ bank["U"].T

def val_rmse(bank, scs, persistence=False):
    err = []
    for sc in scs:
        n0, n1 = sc.n_c, sc.n_end
        if n1 - n0 <= C.Kh: continue
        Xh = sc.Xn[n0:n1 - C.Kh + 1] if persistence else forecast(bank, sc.Xn[n0:n1 - C.Kh + 1], sc.l_hat, C.Kh)
        err.append(((Xh - sc.Xn[n0 + C.Kh:n1 + 1]) ** 2).mean(1))
    return float(np.sqrt(np.concatenate(err).mean()))

t0 = time.time(); cand = []
for r in C.r_cands:
    for lam in C.ridge_grid:                                           # r and the ridge weight are chosen on validation forecasts
        bk = build_bank(r, lam); cand.append((r, lam, val_rmse(bk, nc_val), bk))
emin = min(c[2] for c in cand)
elig = [c for c in cand if c[0] >= r_energy and c[2] <= (1 + C.r_tol) * emin]
if not elig:                                                           # energy level not reachable within candidates
    print(f"note: no candidate reaches the {C.energy:.1%} energy level ({r_energy} modes); using the error criterion alone")
    elig = [c for c in cand if c[2] <= (1 + C.r_tol) * emin]
r_star, lam_star, e_star, bank = sorted(elig, key=lambda c: (c[0], -c[1]))[0]
bank["eps"] = max(e_star, 1e-6)                                        # RMS Kh-step error of nominal recovery (Eq. 31)
errs = {(c[0], c[1]): round(c[2], 4) for c in cand}
print("persistence baseline RMSE:", round(val_rmse(bank, nc_val, persistence=True), 4))
print("validation RMSE by (r, ridge):", errs, "-> r* =", r_star, "ridge* =", lam_star,
      f"| class-specific operators: {bank['n_class_models']}/{len(pairs)} (rest pooled) | {time.time()-t0:.1f}s")
sr = [np.max(np.abs(np.linalg.eigvals(K))) for K in list(bank["Ks"].values())[:10]]
print("spectral radius of first class operators:", np.round(sr, 3))

def koopman_outputs(sc):
    """Eqs. (31)-(32): standardised mismatch e_n and projected physical state -> stress / affinity inputs."""
    T = len(sc.t); Xh = forecast(bank, sc.Xn, sc.l_hat, C.Kh)            # row n = forecast of sample n+Kh issued at n
    e = np.full(T, np.nan); n = np.arange(sc.n_c + C.Kh, T)
    e[n] = np.sqrt(((sc.Xn[n] - Xh[n - C.Kh]) ** 2).mean(1)) / bank["eps"]
    Xp_hat = G.mu + G.sd * Xh                                           # back to physical units (Eq. 9 inverse)
    Sh = np.hypot(Xp_hat[:, SL["P"]], Xp_hat[:, SL["Q"]])
    sc.e_mis = e; sc.rho_hat = Sh / (G.S_ref[None, :] + C.eps_s); sc.V_hat = Xp_hat[:, SL["V"]]
for sc in train_u + val_u + test_u: koopman_outputs(sc)
print("Koopman mismatch computed (median e on stable train scenarios:",
      np.round(np.nanmedian(np.concatenate([s.e_mis for s in nc_train])), 2), ")")

## 7. SPIL — forecast-conditioned spectral perturbation localisation (Eqs. 33–36, Prop. 3)

In [ ]:
# ===== Sec. 3.5  SPIL: forecast-conditioned spectral perturbation localisation =====
def affinity(rho, V, status):
    """Eq. (14): affinity decreases with voltage depression and above-reference stress; 0 for open lines."""
    vf = np.minimum(1, V[:, EI] / G.V_ref[EI]) * np.minimum(1, V[:, EJ] / G.V_ref[EJ])
    a = np.exp(-C.beta_aff * np.maximum(0, rho - 1)) * np.clip(vf, 1e-3, 1)
    return np.where(status > 0, np.maximum(a, 1e-6), 0.0)

_cs = {}
def comp_struct(status):
    key = status.astype(np.int8).tobytes()
    if key not in _cs:
        lab = comp_labels(status); out = []
        for c in np.unique(lab):
            nodes = np.flatnonzero(lab == c)
            if len(nodes) < 2: continue                               # isolated nodes carry no connectivity
            loc = {int(n): i for i, n in enumerate(nodes)}
            eidx = np.flatnonzero((status > 0) & np.isin(EI, nodes))
            out.append((nodes, eidx, np.array([loc[EIl[e]] for e in eidx]), np.array([loc[EJl[e]] for e in eidx])))
        _cs[key] = out
    return _cs[key]

def spectral(a, status):
    """Eqs. (33)-(34): size-weighted algebraic connectivity and eigenspace-averaged edge-sensitivity shares."""
    lam_bar, q = 0.0, np.zeros(NL)
    for nodes, eidx, li, lj in comp_struct(status):
        n = len(nodes); w = a[eidx]; A = np.zeros((n, n)); A[li, lj] = w; A[lj, li] = w
        ev, U = np.linalg.eigh(np.diag(A.sum(1)) - A)
        lam = ev[1]
        if lam <= 1e-10: continue
        idx = np.flatnonzero(np.abs(ev - lam) <= C.eig_tol * lam); idx = idx[idx >= 1]      # repeated eigenvalue -> projector
        d = U[li][:, idx] - U[lj][:, idx]
        q[eidx] = (n / N) * w * (d ** 2).sum(1) / (len(idx) * lam)
        lam_bar += (n / N) * lam
    return lam_bar, q

# ---- numerical check of Proposition 3 on random affinities
rr = np.random.default_rng(2); ok_sum = ok_lb = True; ok_weyl = True
for _ in range(100):
    a = rr.uniform(0.2, 2.0, NL); st = np.ones(NL, np.int8); lam, q = spectral(a, st)
    ok_sum &= abs(q.sum() - 1) < 1e-8                                    # (i) shares sum to 1
    for l in rr.choice(NL, 4, replace=False):                           # (ii) first-order score never overstates exact loss
        s2 = st.copy(); s2[l] = 0; A2 = np.zeros((N, N)); A2[EI, EJ] = a * s2; A2[EJ, EI] = a * s2
        lam2 = np.linalg.eigvalsh(np.diag(A2.sum(1)) - A2)[1]            # exact lambda_2 (0 if the removal disconnects)
        ok_lb &= (lam - lam2) >= q[l] * lam - 1e-9
    da = rr.uniform(-0.1, 0.1, NL); lam3, _ = spectral(np.maximum(a + da, 1e-3), st)   # (iii) Weyl-type stability
    ok_weyl &= abs(lam3 - lam) <= 2 * np.abs(da).max() * 2 + 1e-6      # ||dL||_2 <= 2 * max-degree-change bound (loose, see Eq. 36)
print(f"Proposition 3 checks: shares sum to 1: {ok_sum} | lower bound holds: {ok_lb} | eigenvalue stability: {ok_weyl}")

def spectral_features(sc):
    T = len(sc.t); sc.lam = np.full(T, np.nan); sc.lam_p = np.full(T, np.nan); sc.q_hat = np.zeros((T, NL))
    a_c = affinity(sc.rho, sc.V, sc.status); a_p = affinity(sc.rho_hat, sc.V_hat, sc.status)
    for n in range(sc.n_c, sc.n_end + 1):
        sc.lam[n], _ = spectral(a_c[n], sc.status[n]); sc.lam_p[n], sc.q_hat[n] = spectral(a_p[n], sc.status[n])

t0 = time.time()
for sc in train_u + val_u + test_u: spectral_features(sc)
print(f"SPIL features computed in {time.time()-t0:.1f}s")

## 8. TDS-Risk — evidence, valid origins/targets (Eq. 3), scenario-balanced logistic heads (Eqs. 6, 37, 38)

In [ ]:
# ===== Sec. 3.6  TDS-Risk: evidence (Eq. 37), valid origins & targets (Eq. 3), risk heads (Eqs. 6, 38) =====
EV_NAMES = ([f"PHTS_{k}" for k in ["D0", "D1", "drift0", "drift1"]] + [f"PHTS_db0_{g}" for g in range(C.G)] +
            [f"PHTS_db1_{g}" for g in range(C.G)] + ["KOOP_log1p_mismatch", "KOOP_max_proj_stress",
            "SPIL_lambda", "SPIL_lambda_proj", "SPIL_weakening"])
COLS = dict(PHTS=np.arange(0, 4 + 2 * C.G), KOOP=np.array([4 + 2 * C.G, 5 + 2 * C.G]), SPIL=np.arange(6 + 2 * C.G, 9 + 2 * C.G))

def evidence(sc, z=None):
    z = sc.z_phts if z is None else z
    dyn = np.c_[np.log1p(sc.e_mis), np.minimum(np.where(sc.status > 0, sc.rho_hat, 0).max(1), 10.0)]
    spec = np.c_[sc.lam, sc.lam_p, np.maximum(sc.lam - sc.lam_p, 0)]
    return np.hstack([z, dyn, spec])

def stress_stats(sc):
    """Scalar stress statistics only (no topology): control for the claim that PHTS adds more than stress summaries."""
    r = np.where(sc.status > 0, sc.rho, np.nan)
    with np.errstate(all="ignore"):
        out = np.c_[np.nanmean(r, 1), np.nanmax(r, 1), np.nanstd(r, 1), np.nanmedian(r, 1), np.nanquantile(r, 0.9, axis=1),
                    sc.status.sum(1), (sc.V / G.V_ref[None, :]).min(1)]
    return np.nan_to_num(out, nan=0.0)

def build_origins(sc, E):
    T = len(sc.t); n = np.arange(T); ok = np.zeros(T, bool)
    if sc.ref is not None:
        n0 = max(C.n_hist, sc.n_c, sc.ref[1], sc.n_c + C.Kh)                      # clearance, reference and forecast available
        ok = ((n >= n0) & (sc.t <= sc.t_c_hat + C.Tmax) & (sc.t < sc.tau) & (sc.t + C.Th <= sc.t[-1]) & np.isfinite(E).all(1))
    sc.orig = np.flatnonzero(ok); tn = sc.t[sc.orig]
    sc.y_sys = (sc.tau <= tn + C.Th).astype(float)                                   # any qualifying event within Th
    Y = np.zeros((len(tn), NL))
    for l, te in sc.line_events: Y[(tn < te) & (te <= tn + C.Th), l] = 1               # monitored-line outage of line l within Th
    sc.Y = Y; sc.y_line = Y.any(1).astype(float)

for sc in train_u + val_u + test_u:
    sc.E = evidence(sc); sc.SS = stress_stats(sc); build_origins(sc, sc.E)

def stack(scs, field="E", cols=None):
    Xs, ys, yl, ws, ids = [], [], [], [], []
    for i, sc in enumerate(scs):
        if len(sc.orig) == 0: continue
        M = getattr(sc, field)[sc.orig]; Xs.append(M if cols is None else M[:, cols])
        ys.append(sc.y_sys); yl.append(sc.y_line); ws.append(np.full(len(sc.orig), 1.0 / len(sc.orig))); ids.append(np.full(len(sc.orig), i))
    X = np.vstack(Xs); w = np.concatenate(ws); w = w / w.mean()                      # Eq. (6): each scenario has equal total weight
    return X, np.concatenate(ys), np.concatenate(yl), w, np.concatenate(ids)

Xtr, ys_tr, yl_tr, w_tr, _ = stack(train_u); Xva, ys_va, yl_va, w_va, _ = stack(val_u); Xte, ys_te, yl_te, w_te, id_te = stack(test_u)
mu_e, sd_e = Xtr.mean(0), Xtr.std(0); sd_e = np.where(sd_e < 1e-9, 1.0, sd_e)       # Eq. (37), training statistics only
std_ = lambda X: np.clip((X - mu_e) / sd_e, -5, 5)           # clip: numerical safeguard against heavy-tailed evidence
print(f"origins  train {len(Xtr)} (pos sys {ys_tr.mean():.3f}, line {yl_tr.mean():.3f}) | val {len(Xva)} | test {len(Xte)} "
      f"(pos sys {ys_te.mean():.3f}, line {yl_te.mean():.3f}) | evidence dim {Xtr.shape[1]}")

def fit_head(Xa, ya, wa, Xb, yb, wb, cols=None):
    """Eq. (6): scenario-balanced ridge logistic loss; gamma and optional class weights chosen on validation loss."""
    cols = slice(None) if cols is None else cols; best = None
    for g in C.gamma_grid:
        for cw in (None, "balanced"):
            m = LogisticRegression(C=1.0 / g, class_weight=cw, max_iter=5000).fit(Xa[:, cols], ya, sample_weight=wa)
            p = np.clip(m.predict_proba(Xb[:, cols])[:, 1], 1e-6, 1 - 1e-6)
            ll = -(wb * (yb * np.log(p) + (1 - yb) * np.log(1 - p))).sum() / wb.sum()
            if best is None or ll < best[0] - 1e-9: best = (ll, m, cw, g)
    return best[1], (best[2], best[3]), best[0]

head_sys, cfg_s, ll_s = fit_head(std_(Xtr), ys_tr, w_tr, std_(Xva), ys_va, w_va)
head_line, cfg_l, ll_l = fit_head(std_(Xtr), yl_tr, w_tr, std_(Xva), yl_va, w_va)
print(f"system head: (class_weight, gamma)={cfg_s}, val weighted log-loss {ll_s:.4f} | line head: {cfg_l}, val log-loss {ll_l:.4f}")
for sc in train_u + val_u + test_u:
    if len(sc.orig):
        Z = std_(sc.E[sc.orig]); sc.p_sys = head_sys.predict_proba(Z)[:, 1]; sc.p_line = head_line.predict_proba(Z)[:, 1]
    else: sc.p_sys = sc.p_line = np.zeros(0)
coef = pd.Series(head_sys.coef_[0], index=EV_NAMES).sort_values(key=np.abs, ascending=False)
print("largest system-head weights (standardised evidence):"); print(coef.head(8).round(3).to_string())

## 9. Alarm, lead time and threshold calibration (Eqs. 5, 7) and test evaluation

In [ ]:
# ===== Eqs. (5) and (7): alarm, warning lead time and threshold calibration on VALIDATION scenarios only =====
def first_alarm(sc, p, theta):
    h = np.flatnonzero(p >= theta)
    return float(sc.t[sc.orig[h[0]]]) if h.size else np.nan

def alarm_stats(scs, theta, attr="p_sys"):
    s = dict(n_stable=0, fa=0, n_casc=0, pre=0, det=0, leads=[])
    for sc in scs:
        ta = first_alarm(sc, getattr(sc, attr), theta) if len(sc.orig) else np.nan
        if np.isinf(sc.tau):
            s["n_stable"] += 1; s["fa"] += int(not np.isnan(ta))
        else:
            s["n_casc"] += 1
            if not np.isnan(ta):
                if ta < sc.tau - C.Th: s["pre"] += 1                                  # premature: outside the credited window
                else: s["det"] += 1; s["leads"].append(sc.tau - ta)                    # Eq. (5): credited warning
    return s

def summarize(s):
    nc, ns = max(s["n_casc"], 1), max(s["n_stable"], 1)
    return dict(false_alarm=s["fa"] / ns, premature=s["pre"] / nc, detection=s["det"] / nc,
                mean_lead=float(np.sum(s["leads"]) / nc), median_lead_detected=float(np.median(s["leads"])) if s["leads"] else 0.0)

rows = []
for th in C.theta_grid:
    r = summarize(alarm_stats(val_u, th)); r["theta"] = th; rows.append(r)
cal = pd.DataFrame(rows)
feas = cal[(cal.false_alarm <= C.alpha_f) & (cal.premature <= C.alpha_p) & (cal.detection >= C.delta_det)]
if len(feas):
    theta_star = float(feas.sort_values("mean_lead", ascending=False).iloc[0].theta); how = "feasible under all three bounds"
else:                                                                                   # least-violating fallback, reported openly
    viol = (cal.false_alarm - C.alpha_f).clip(lower=0) + (cal.premature - C.alpha_p).clip(lower=0) + (C.delta_det - cal.detection).clip(lower=0)
    cal["violation"] = viol; theta_star = float(cal.sort_values(["violation", "mean_lead"], ascending=[True, False]).iloc[0].theta)
    how = "NO threshold satisfies all bounds on the validation scenarios -> least-violating threshold"
print(f"theta* = {theta_star} ({how})")
print(cal[cal.theta == theta_star].round(3).to_string(index=False))

# ===== Test evaluation with the frozen threshold =====
res = summarize(alarm_stats(test_u, theta_star)); st_te = alarm_stats(test_u, theta_star)
print(f"\nTEST (theta frozen at {theta_star}): stable scenarios {st_te['n_stable']}, cascading {st_te['n_casc']}")
print({k: round(v, 3) for k, v in res.items()})
# cascades whose onset precedes the reference have an empty origin set and count as missed (Eqs. 3, 5, 7)
miss_empty = sum(1 for s in test_u if np.isfinite(s.tau) and len(s.orig) == 0)
print("cascading test scenarios with empty origin set (onset before reference/forecast ready):", miss_empty)

def origin_metrics(y, p):
    return dict(AUROC=roc_auc_score(y, p), AP=average_precision_score(y, p), Brier=brier_score_loss(y, p), prevalence=y.mean())
pte_sys = head_sys.predict_proba(std_(Xte))[:, 1]; pte_line = head_line.predict_proba(std_(Xte))[:, 1]
print("origin-level system head :", {k: round(v, 4) for k, v in origin_metrics(ys_te, pte_sys).items()})
print("origin-level line head   :", {k: round(v, 4) for k, v in origin_metrics(yl_te, pte_line).items()})

## 10. Line vulnerability ranking (Eq. 39)

In [ ]:
# ===== Eq. (39): line vulnerability ranking, evaluated on origins with a qualifying future line outage =====
def rank_metrics(score, truth, Ks=(1, 3, 5)):
    out = {}
    if score.max() <= 0:                                                  # "unlocalised indication": no ranking returned
        return {f"{m}@{k}": 0.0 for k in Ks for m in ("P", "R", "NDCG")}
    order = np.argsort(-score, kind="stable")
    for k in Ks:
        top = order[:k]; hits = truth[top]
        dcg = (hits / np.log2(np.arange(2, k + 2))).sum(); idcg = (1 / np.log2(np.arange(2, min(int(truth.sum()), k) + 2))).sum()
        out[f"P@{k}"] = hits.sum() / k; out[f"R@{k}"] = hits.sum() / truth.sum(); out[f"NDCG@{k}"] = dcg / idcg
    return out

def vulnerability_scores(sc, n):
    act = sc.status[n]; w = np.minimum(sc.rho_hat[n], C.smax)
    c = connectivity_impacts(w, act) * act                                # projected stresses, current status
    ratio = c / c.sum() if c.sum() > 0 else np.zeros(NL)
    full = (w / C.smax) * sc.q_hat[n] * ratio * act                       # Eq. (39)
    return dict(full=full, stress_cur=np.minimum(sc.rho[n], C.smax) * act, stress_proj=w * act,
                share_proj=sc.q_hat[n] * act, stress_x_share=(w / C.smax) * sc.q_hat[n] * act)

rr = np.random.default_rng(3); pool = []
for sc in test_u:
    for j in np.flatnonzero(sc.y_line > 0): pool.append((sc, j))
sel = [pool[i] for i in rr.choice(len(pool), min(len(pool), 1500), replace=False)] if pool else []
acc = {}
for sc, j in sel:
    n = sc.orig[j]; S = vulnerability_scores(sc, n); truth = sc.Y[j]
    S["random"] = (rr.random(NL) * sc.status[n]); 
    for name, sco in S.items():
        for k, v in rank_metrics(sco, truth).items(): acc.setdefault(name, {}).setdefault(k, []).append(v)
rank_tab = pd.DataFrame({m: {k: np.mean(v) for k, v in d.items()} for m, d in acc.items()}).T
print(f"Line ranking on {len(sel)} test origins with a future line outage (truth = lines out within Th):")
print(rank_tab.round(3).to_string())
print("share of origins where Eq. (39) is all-zero (unlocalised):",
      round(np.mean([vulnerability_scores(sc, sc.orig[j])['full'].max() <= 0 for sc, j in sel[:300]]), 3) if sel else None)

## 11. Ablations and edge-permutation test

In [ ]:
# ===== Ablations: evidence streams, stress-statistics control, and the edge-permutation test (Prop. 2) =====
for sc in train_u + val_u + test_u:                      # PHTS recomputed with the SAME stress values re-assigned to random lines
    kap = np.random.default_rng(abs(hash(sc.sid)) % (2 ** 31)).random(NL)
    sc.E_perm = evidence(sc, phts_signature(sc, kappa=kap))

def fit_eval(field, cols=None):
    Xa, ya, _, wa, _ = stack(train_u, field, cols); Xb, yb, _, wb, _ = stack(val_u, field, cols); Xc, yc, _, _, _ = stack(test_u, field, cols)
    mu, sd = Xa.mean(0), Xa.std(0); sd = np.where(sd < 1e-9, 1.0, sd); f = lambda X: np.clip((X - mu) / sd, -5, 5)
    head, cfg, _ = fit_head(f(Xa), ya, wa, f(Xb), yb, wb)
    m = origin_metrics(yc, head.predict_proba(f(Xc))[:, 1]); m["(cw, gamma)"] = str(cfg); return m

P, K_, S_ = COLS["PHTS"], COLS["KOOP"], COLS["SPIL"]
variants = {"stress statistics only (no topology)": ("SS", None), "PHTS only": ("E", P), "Koopman only": ("E", K_), "SPIL only": ("E", S_),
            "PHTS + Koopman": ("E", np.r_[P, K_]), "PHTS + SPIL": ("E", np.r_[P, S_]), "Koopman + SPIL": ("E", np.r_[K_, S_]),
            "full TDS-Risk": ("E", None), "full, PHTS edge-permuted": ("E_perm", None), "PHTS only, edge-permuted": ("E_perm", P)}
abl = pd.DataFrame({k: fit_eval(*v) for k, v in variants.items()}).T
print(abl[["AUROC", "AP", "Brier", "(cw, gamma)"]].to_string(float_format=lambda x: f"{x:.3f}"))
print("\nIf 'PHTS only' does not clearly beat 'PHTS only, edge-permuted', the topological signature carries no information beyond"
      " scalar stress statistics on this data (Proposition 2).")

## 12. Measured online runtime (Sec. 3.7)

In [ ]:
# ===== Runtime of the online pipeline (Sec. 3.7): measured, not asserted =====
sc = max(test_u, key=lambda s: len(s.orig)); n_list = sc.orig[:min(200, len(sc.orig))]
i0, i1 = sc.ref; cur = [betti_curves(sc.rho[n], sc.status[n]) for n in range(i0, i1 + 1)]; B0r = np.mean([c[0] for c in cur], 0); B1r = np.mean([c[1] for c in cur], 0)
tm = dict(stress_status_affinity=0, phts=0, koopman=0, spil=0, head=0)
for n in n_list:
    t = time.perf_counter(); a = affinity(sc.rho[n:n + 1], sc.V[n:n + 1], sc.status[n:n + 1])[0]; tm["stress_status_affinity"] += time.perf_counter() - t
    t = time.perf_counter(); b0, b1 = betti_curves(sc.rho[n], sc.status[n]); np.maximum(b0 - B0r, 0).sum(); np.maximum(B1r - b1, 0).sum(); tm["phts"] += time.perf_counter() - t
    t = time.perf_counter(); Xh = forecast(bank, sc.Xn[n:n + 1], sc.l_hat, C.Kh); tm["koopman"] += time.perf_counter() - t
    t = time.perf_counter(); spectral(a, sc.status[n]); spectral(a, sc.status[n]); tm["spil"] += time.perf_counter() - t
    t = time.perf_counter(); head_sys.predict_proba(std_(sc.E[n:n + 1])); tm["head"] += time.perf_counter() - t
tm = {k: 1e3 * v / len(n_list) for k, v in tm.items()}
print({k: round(v, 3) for k, v in tm.items()}, "ms/sample")
print(f"total {sum(tm.values()):.2f} ms per sample vs sampling interval {C.dt*1e3:.0f} ms "
      f"(paper PMU rate 10 ms) -> real-time at dt={C.dt}s: {sum(tm.values()) < C.dt*1e3}; at 10 ms: {sum(tm.values()) < 10}"
      " [single-threaded numpy on this machine; the 10 ms claim must be re-measured on the target hardware]")

## 13. Figures

In [ ]:
# ===== Figures =====
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
# (a) Betti curves: reference vs deteriorated state of a cascading test scenario
cs_ = [s for s in test_u if np.isfinite(s.tau) and len(s.orig)]
if cs_:
    sc = cs_[0]; i0, i1 = sc.ref; cur = [betti_curves(sc.rho[n], sc.status[n]) for n in range(i0, i1 + 1)]
    n_late = sc.orig[-1]; b0, b1 = betti_curves(sc.rho[n_late], sc.status[n_late])
    ax[0, 0].step(KG, np.mean([c[0] for c in cur], 0), label="beta0 reference", color="C0"); ax[0, 0].step(KG, b0, "--", label="beta0 late", color="C0")
    ax[0, 0].step(KG, np.mean([c[1] for c in cur], 0), label="beta1 reference", color="C3"); ax[0, 0].step(KG, b1, "--", label="beta1 late", color="C3")
    ax[0, 0].set(title=f"PHTS Betti curves ({sc.sid}, onset {sc.tau:.1f}s)", xlabel="stress level s", ylabel="Betti number"); ax[0, 0].legend(fontsize=7)
    # (b) risk trajectory with alarm
    tt = sc.t[sc.orig]; ax[0, 1].plot(tt, sc.p_sys, label="system risk"); ax[0, 1].axhline(theta_star, color="k", ls=":", label="theta*")
    ax[0, 1].axvline(sc.tau, color="r", label="onset"); ax[0, 1].axvspan(sc.tau - C.Th, sc.tau, color="r", alpha=0.1)
    ta = first_alarm(sc, sc.p_sys, theta_star); 
    if not np.isnan(ta): ax[0, 1].axvline(ta, color="g", ls="--", label="first alarm")
    ax[0, 1].set(title="TDS-Risk on a cascading scenario", xlabel="time [s]", ylim=(0, 1)); ax[0, 1].legend(fontsize=7)
    # (c) evidence streams
    nn = np.arange(sc.n_c + C.Kh, sc.n_end + 1); ax[0, 2].plot(sc.t[nn], sc.lam[nn], label="lambda (current)"); ax[0, 2].plot(sc.t[nn], sc.lam_p[nn], label="lambda (projected)")
    ax2 = ax[0, 2].twinx(); ax2.plot(sc.t[nn], np.log1p(sc.e_mis[nn]), "C3", alpha=.6, label="log(1+mismatch)"); ax2.grid(False)
    ax[0, 2].axvline(sc.tau, color="r"); ax[0, 2].set(title="SPIL connectivity and Koopman mismatch", xlabel="time [s]"); ax[0, 2].legend(fontsize=7, loc="lower left")
# (d) ROC / reliability on test origins
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(ys_te, pte_sys); ax[1, 0].plot(fpr, tpr); ax[1, 0].plot([0, 1], [0, 1], "k:"); ax[1, 0].set(title=f"ROC (test origins, AUROC {roc_auc_score(ys_te, pte_sys):.3f})", xlabel="FPR", ylabel="TPR")
bins = np.linspace(0, 1, 11); b = np.clip(np.digitize(pte_sys, bins) - 1, 0, 9)
mp = [pte_sys[b == k].mean() if (b == k).any() else np.nan for k in range(10)]; fr = [ys_te[b == k].mean() if (b == k).any() else np.nan for k in range(10)]
ax[1, 1].plot(mp, fr, "o-"); ax[1, 1].plot([0, 1], [0, 1], "k:"); ax[1, 1].set(title="Reliability diagram (system head)", xlabel="predicted risk", ylabel="observed frequency")
# (e) lead times
leads = alarm_stats(test_u, theta_star)["leads"]
ax[1, 2].hist(leads, bins=10) if leads else ax[1, 2].text(0.3, 0.5, "no credited warnings")
ax[1, 2].set(title=f"Warning lead times (test, {len(leads)} credited)", xlabel="lead time [s]"); plt.tight_layout(); plt.show()

if len(rank_tab):
    rank_tab[["NDCG@1", "NDCG@3", "NDCG@5"]].plot.bar(figsize=(8, 3.5), title="Line ranking NDCG (test origins with a future line outage)"); plt.tight_layout(); plt.show()
abl["AUROC"].plot.barh(figsize=(8, 4), title="Ablation: origin-level AUROC (test)"); plt.axvline(0.5, color="k", ls=":"); plt.tight_layout(); plt.show()